In [1]:
from utils.random_forest_utils.rf_preprocessing_utils import WindowAlgPreprocessor

rf_preprocessor_clamping = WindowAlgPreprocessor(sensors_path="../../../../data/ml/label_df/clamping_features.csv", target_path="../../../../data/ml/targets.csv")
sensors_df_clamping, target_df = rf_preprocessor_clamping.read_data()
sensors_df_clamping = rf_preprocessor_clamping.feature_selection()
rf_preprocessor_clamping.normalize_angle()
rf_preprocessor_bending = WindowAlgPreprocessor(sensors_path="../../../../data/ml/label_df/bending_features.csv", target_path="../../../../data/ml/targets.csv")
sensors_df_bending, _ = rf_preprocessor_bending.read_data()
sensors_df_bending = rf_preprocessor_bending.feature_selection()
rf_preprocessor_bending.normalize_angle()
rf_preprocessor_declamping = WindowAlgPreprocessor(sensors_path="../../../../data/ml/label_df/declamping_features.csv", target_path="../../../../data/ml/targets.csv")
sensors_df_declamping, _ = rf_preprocessor_declamping.read_data()
sensors_df_declamping = rf_preprocessor_declamping.feature_selection()
rf_preprocessor_declamping.normalize_angle()

,Experiment_ID,Angle[degree]ORDistance[mm],Secondary-axis [mm],Main-axis [mm],Out-of-roundness [-],Collapse [mm]
0,2,0.000000,0.900079,0.313091,0.999954,0.313091
1,2,0.022017,0.908296,0.425584,0.834481,0.425584
2,2,0.044033,0.905549,0.591178,0.582994,0.591178
3,2,0.066050,0.901145,0.751356,0.338801,0.751356
4,2,0.088067,0.909703,0.867774,0.167582,0.867774
...,...,...,...,...,...,...
14558,318,0.902686,0.928583,0.029142,0.997804,0.029142
14559,318,0.924703,0.922565,0.025609,1.000000,0.025609
14560,318,0.946720,0.921322,0.032466,0.992607,0.032466
14561,318,0.968736,0.925369,0.051137,0.974276,0.051137


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis, entropy

def resample_experiment_fast(group, n=46, metric='mean'):
    """
    Optimized resampling function using vectorized operations.
    Up to 10-100x faster than the original implementation.
    """
    # Sort by time
    group = group.sort_values('Time_[s]')
    time_col = group['Time_[s]'].values
    
    # Assign each row to a time bin
    time_bins = np.linspace(time_col.min(), time_col.max(), n + 1)
    bin_indices = np.digitize(time_col, time_bins[:-1]) - 1
    bin_indices = np.clip(bin_indices, 0, n - 1)
    
    # Get experiment ID
    exp_id = group['Experiment_ID'].iloc[0]
    
    # Select numeric columns only
    cols_to_process = [col for col in group.columns 
                       if col not in ['Time_[s]', 'Experiment_ID']]
    
    results = []
    
    # Process each bin
    for bin_idx in range(n):
        mask = bin_indices == bin_idx
        if not mask.any():
            continue
            
        row_data = {'Experiment_ID': exp_id}
        
        for col in cols_to_process:
            values = group[col].values[mask]
            if len(values) == 0:
                continue
            
            # Compute metric using vectorized operations
            if metric == 'mean':
                row_data[f'{col}_mean'] = values.mean()
            elif metric == 'median':
                row_data[f'{col}_median'] = np.median(values)
            elif metric == 'min':
                row_data[f'{col}_min'] = values.min()
            elif metric == 'max':
                row_data[f'{col}_max'] = values.max()
            elif metric == 'range':
                row_data[f'{col}_range'] = values.ptp()
            elif metric == 'std':
                row_data[f'{col}_std'] = values.std()
            elif metric == 'var':
                row_data[f'{col}_var'] = values.var()
            elif metric == 'mad':
                row_data[f'{col}_mad'] = np.abs(values - values.mean()).mean()
            elif metric == 'rms':
                row_data[f'{col}_rms'] = np.sqrt((values ** 2).mean())
            elif metric == 'skew':
                row_data[f'{col}_skew'] = skew(values)
            elif metric == 'kurtosis':
                row_data[f'{col}_kurtosis'] = kurtosis(values)
            elif metric == 'energy':
                row_data[f'{col}_energy'] = (values ** 2).sum()
            elif metric == 'entropy':
                abs_vals = np.abs(values)
                probs = abs_vals / (abs_vals.sum() + 1e-12)
                row_data[f'{col}_entropy'] = entropy(probs + 1e-12)
            elif metric == 'cv':
                row_data[f'{col}_cv'] = values.std() / (values.mean() + 1e-12)
            elif metric == 'iqr':
                row_data[f'{col}_iqr'] = np.percentile(values, 75) - np.percentile(values, 25)
            elif metric == 'p25':
                row_data[f'{col}_p25'] = np.percentile(values, 25)
            elif metric == 'p75':
                row_data[f'{col}_p75'] = np.percentile(values, 75)
            elif metric == 'trend_slope':
                if len(values) > 1:
                    row_data[f'{col}_trend_slope'] = np.polyfit(np.arange(len(values)), values, 1)[0]
                else:
                    row_data[f'{col}_trend_slope'] = 0
        
        results.append(row_data)
    
    return pd.DataFrame(results)


# Alternative: Ultra-fast version using pandas groupby (even faster for 'mean', 'std', 'min', 'max')
def resample_experiment_ultrafast(group, n=46, metric='mean'):
    """
    Ultra-optimized version using pandas groupby operations.
    Works best for basic metrics like mean, std, min, max, median.
    """
    group = group.sort_values('Time_[s]')
    time_col = group['Time_[s]'].values
    
    # Assign bins
    time_bins = np.linspace(time_col.min(), time_col.max(), n + 1)
    group['_bin'] = np.digitize(time_col, time_bins[:-1]) - 1
    group['_bin'] = group['_bin'].clip(0, n - 1)
    
    # Select columns to aggregate
    cols_to_agg = [col for col in group.columns 
                   if col not in ['Time_[s]', 'Experiment_ID', '_bin']]
    
    # Map metric to pandas aggregation function
    agg_func_map = {
        'mean': 'mean',
        'median': 'median',
        'min': 'min',
        'max': 'max',
        'std': 'std',
        'var': 'var',
        'sum': 'sum'
    }
    
    if metric in agg_func_map:
        # Use fast pandas groupby
        result = group.groupby('_bin')[cols_to_agg].agg(agg_func_map[metric])
        result = result.add_suffix(f'_{metric}')
        result['Experiment_ID'] = group['Experiment_ID'].iloc[0]
        return result.reset_index(drop=True)
    else:
        # Fall back to custom implementation
        return resample_experiment_fast(group.drop('_bin', axis=1), n, metric)


In [3]:
import matplotlib.pyplot as plt
from ipywidgets import widgets, VBox, interactive_output
from IPython.display import display

def interactive_sensor_target_plot(
    bending_df, clamping_df, declamping_df, target_df,
    target_cols=["Secondary-axis [mm]", "Main-axis [mm]", "Out-of-roundness [-]", "Collapse [mm]"]
):
    """
    Create an interactive plot of sensor data and target data for different datasets and experiments.

    Parameters:
    - bending_df: DataFrame for bending sensors
    - clamping_df: DataFrame for clamping sensors
    - declamping_df: DataFrame for declamping sensors
    - target_df: DataFrame for target values
    - target_cols: list of target columns to plot (default 4 main targets)
    """
    
    dfs = {
        "Bending": bending_df,
        "Clamping": clamping_df,
        "Declamping": declamping_df
    }

    def plot_all(dataset_name, experiment_id):
        df = dfs[dataset_name]
        sensor_cols = [c for c in df.columns if c != "Experiment_ID"]

        # Filter
        if experiment_id == "All":
            sensor_data = df
            target_data = target_df
        else:
            sensor_data = df[df["Experiment_ID"] == experiment_id]
            target_data = target_df[target_df["Experiment_ID"] == experiment_id]

        # --- Subplots ---
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10))

        # ===== TOP SENSOR PLOT =====
        for col in sensor_cols:
            ax1.plot(sensor_data.index, sensor_data[col], label=col, linewidth=1)

        ax1.set_title(f"{dataset_name} Sensors vs Time (Experiment {experiment_id})")
        ax1.set_xlabel("Time [s]")
        ax1.set_ylabel("Sensor Values")
        ax1.grid(True)
        ax1.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)

        # ===== TARGET PLOT =====
        for col in target_cols:
            if col in target_data.columns:
                ax2.plot(target_data["Angle[degree]ORDistance[mm]"], target_data[col], label=col, linewidth=1)

        ax2.set_title("Target Data")
        ax2.set_xlabel("Angle [degree] / Distance [mm]")
        ax2.set_ylabel("Value")
        ax2.grid(True)
        ax2.legend(loc="center left", bbox_to_anchor=(1, 0.5))

        plt.tight_layout()
        plt.show()

    # === Widgets ===
    dataset_dropdown = widgets.Dropdown(options=list(dfs.keys()), description="Dataset:")
    exp_dropdown = widgets.Dropdown(options=["All"], description="Experiment:")

    # Dynamically update available experiment IDs
    def update_exp_list(*args):
        df = dfs[dataset_dropdown.value]
        with exp_dropdown.hold_trait_notifications():
            exp_dropdown.options = ["All"] + sorted(df["Experiment_ID"].unique())

    dataset_dropdown.observe(update_exp_list, names="value")
    update_exp_list()

    # === Use interactive_output instead of interact ===
    output = interactive_output(
        plot_all,
        {"dataset_name": dataset_dropdown, "experiment_id": exp_dropdown}
    )

    # Display clean layout
    display(VBox([dataset_dropdown, exp_dropdown, output]))


In [4]:
import seaborn as sns

def interactive_correlation_analysis(
    bending_df, clamping_df, declamping_df, target_df,
    drop_target_cols_containing="Angle"
):
    """
    Prepare sensor and target data, then create interactive correlation plots:
    - Self-correlation heatmap
    - Correlation with targets
    - Top-K correlation bar plots
    """

    # ======== Helper Functions ========
    def prepare_inputs(df, drop_columns=None):
        if df is None or df.empty:
            raise ValueError("Input dataframe cannot be None or empty")
        
        # Numeric only
        df_numeric = df.select_dtypes(include="number")
        
        # Drop Experiment_ID
        experiment_id_cols = [col for col in df_numeric.columns if col.lower() == 'experiment_id']
        if experiment_id_cols:
            df_numeric = df_numeric.drop(columns=experiment_id_cols)
        
        # Drop custom columns
        if drop_columns:
            df_numeric = df_numeric.drop(columns=drop_columns, errors="ignore")
        
        # Ensure unique index
        if not df_numeric.index.is_unique:
            df_numeric = df_numeric.reset_index(drop=True)
        
        return df_numeric

    def plot_correlation_heatmaps(dataset_name, df, targets_numeric):
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # Self-correlation
        corr_self = df.corr()
        sns.heatmap(corr_self, annot=True, fmt=".2f", cmap="coolwarm",
                    ax=axes[0], linewidths=0.5, linecolor='gray', vmin=-1, vmax=1,
                    annot_kws={'size':6})
        axes[0].set_title(f"{dataset_name} Self-Correlation", fontsize=12, fontweight='bold')
        axes[0].tick_params(axis='y', labelsize=8, rotation=0)
        axes[0].tick_params(axis='x', labelsize=8, rotation=45)
        
        # Correlation with targets
        if targets_numeric.empty:
            axes[1].text(0.5,0.5,'No target variables',ha='center',va='center',fontsize=12)
            axes[1].set_title(f"{dataset_name} Correlation with Targets", fontsize=12, fontweight='bold')
        else:
            combined_df = pd.concat([df, targets_numeric], axis=1)
            corr_with_target = combined_df.corr().loc[df.columns, targets_numeric.columns]
            sns.heatmap(corr_with_target, annot=True, fmt=".2f", cmap="coolwarm",
                        ax=axes[1], linewidths=0.5, linecolor='gray', vmin=-1, vmax=1,
                        annot_kws={'size':6})
            axes[1].set_title(f"{dataset_name} Correlation with Targets", fontsize=12, fontweight='bold')
            axes[1].tick_params(axis='y', labelsize=8, rotation=0)
            axes[1].tick_params(axis='x', labelsize=8, rotation=45)

        plt.tight_layout()
        plt.show()

    def plot_top_k_correlations(dataset_name, df, targets_numeric, k=10):
        if targets_numeric.empty:
            return

        combined_df = pd.concat([df, targets_numeric], axis=1)
        corr_with_target = combined_df.corr().loc[df.columns, targets_numeric.columns]

        n_targets = len(targets_numeric.columns)
        ncols = min(3, n_targets)
        nrows = (n_targets + ncols - 1) // ncols

        fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols,4*nrows), squeeze=False)
        axes = axes.flatten()

        for idx, target_col in enumerate(targets_numeric.columns):
            target_corrs = corr_with_target[target_col].dropna()
            top_k_corrs = target_corrs.abs().nlargest(min(k, len(target_corrs)))
            actual_corrs = target_corrs[top_k_corrs.index].sort_values()
            colors = ['#d62728' if x < 0 else '#2ca02c' for x in actual_corrs]

            bars = axes[idx].barh(range(len(actual_corrs)), actual_corrs, color=colors, alpha=0.7, edgecolor='black', linewidth=0.8)
            axes[idx].set_yticks(range(len(actual_corrs)))
            axes[idx].set_yticklabels(actual_corrs.index, fontsize=7)
            axes[idx].set_xlabel('Correlation', fontsize=8)
            axes[idx].set_title(f'{dataset_name} - {target_col} (Top {len(actual_corrs)})', fontsize=9, fontweight='bold')
            axes[idx].axvline(x=0, color='black', linestyle='-', linewidth=0.6)
            axes[idx].grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
            axes[idx].set_xlim([-1,1])
            axes[idx].tick_params(axis='x', labelsize=7)

            for i, (bar,val) in enumerate(zip(bars, actual_corrs)):
                label_x = val+0.03 if val>0 else val-0.03
                ha = 'left' if val>0 else 'right'
                axes[idx].text(label_x, bar.get_y() + bar.get_height()/2, f'{val:.2f}', ha=ha, va='center', fontsize=6)

        # Hide unused axes
        for idx in range(n_targets,len(axes)):
            axes[idx].axis('off')

        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor='#2ca02c', alpha=0.7, label='Positive'),
                           Patch(facecolor='#d62728', alpha=0.7, label='Negative')]
        fig.legend(handles=legend_elements, loc='lower center', ncol=2, fontsize=8, bbox_to_anchor=(0.5,-0.02))

        plt.tight_layout()
        plt.subplots_adjust(bottom=0.05)
        plt.show()

    # ======== Prepare Data ========
    drop_cols = [col for col in target_df.columns if drop_target_cols_containing in col]
    bending_inputs = prepare_inputs(bending_df)
    clamping_inputs = prepare_inputs(clamping_df)
    declamping_inputs = prepare_inputs(declamping_df)
    targets_numeric = prepare_inputs(target_df, drop_columns=drop_cols)

    # Align length
    min_len = min(len(bending_inputs), len(clamping_inputs), len(declamping_inputs), len(targets_numeric))
    bending_inputs, clamping_inputs, declamping_inputs, targets_numeric = (
        bending_inputs.iloc[:min_len],
        clamping_inputs.iloc[:min_len],
        declamping_inputs.iloc[:min_len],
        targets_numeric.iloc[:min_len]
    )

    datasets = {"Bending": bending_inputs, "Clamping": clamping_inputs, "Declamping": declamping_inputs}

    # ======== Interactive Widget ========
    def interactive_combined_plot(dataset, k):
        plot_correlation_heatmaps(dataset, datasets[dataset], targets_numeric)
        plot_top_k_correlations(dataset, datasets[dataset], targets_numeric, k=k)

    widgets.interact(
        interactive_combined_plot,
        dataset=widgets.Dropdown(options=list(datasets.keys()), value='Bending', description='Dataset:'),
        k=widgets.IntSlider(value=10, min=3, max=20, step=1, description='Top K:', continuous_update=False)
    )


In [ ]:
interactive_sensor_target_plot(
    sensors_df_bending,
    sensors_df_clamping,
    sensors_df_declamping,
    target_df
)

In [6]:
interactive_correlation_analysis(
    sensors_df_bending,
    sensors_df_clamping,
    sensors_df_declamping,
    target_df
)

interactive(children=(Dropdown(description='Dataset:', options=('Bending', 'Clamping', 'Declamping'), value='B…

In [7]:


# Option 2: Ultra-fast version (best for mean, std, min, max, median)
df_bending = sensors_df_bending.reset_index()
df_resampled_bending = (
    df_bending.groupby('Experiment_ID', group_keys=False)
    .apply(lambda g: resample_experiment_ultrafast(g, n=40, metric='mean'))
    .reset_index(drop=True)
)


# Apply to all datasets
df_clamping = sensors_df_clamping.reset_index()
df_resampled_clamping = (
    df_clamping.groupby('Experiment_ID', group_keys=False)
    .apply(lambda g: resample_experiment_ultrafast(g, n=40, metric='mean'))
    .reset_index(drop=True)
)



df_declamping = sensors_df_declamping.reset_index()
df_resampled_declamping = (
    df_declamping.groupby('Experiment_ID', group_keys=False)
    .apply(lambda g: resample_experiment_ultrafast(g, n=40, metric='mean'))
    .reset_index(drop=True)
)

/tmp/ipykernel_292882/1813960607.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: resample_experiment_ultrafast(g, n=40, metric='mean'))
/tmp/ipykernel_292882/1813960607.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: resample_experiment_ultrafast(g, n=40, metric='mean'))
/tmp/ipykernel_292882/1813960607.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping co

In [12]:
interactive_sensor_target_plot(
    df_resampled_bending,
    df_resampled_clamping,
    df_resampled_declamping,
    target_df
)

In [9]:
interactive_correlation_analysis(
    df_resampled_bending,
    df_resampled_clamping,
    df_resampled_declamping,
    target_df
)

interactive(children=(Dropdown(description='Dataset:', options=('Bending', 'Clamping', 'Declamping'), value='B…